# Advanced Brain Tumor Analysis: SOTA Training Pipeline

This notebook implements a high-performance, clinically robust training pipeline using **EfficientNet-V2-S**. We employ several advanced techniques including **Mixup**, **CutMix**, **Label Smoothing**, and **OneCycleLR** to maximize generalization and minimize overfitting.

### 🔬 Novel Enhancements:
- **Recursive Data Discovery**: Automatically finds class folders regardless of nesting depth.
- **Hybrid Pooling Head**: Combines Global Average and Global Max pooling to capture both diffuse and focal tumor features.
- **Regularization Suite**: Integrates Mixup and CutMix augmentations for superior robustness.
- **Medical Preprocessing**: Uses CLAHE (Contrast Limited Adaptive Histogram Equalization) for inter-patient normalization.
- **Optimized Convergence**: Uses the OneCycle Learning Rate strategy for efficient training.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import v2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Set device & Seed
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.device('cuda' if torch.cuda.is_available() else 'cpu') else 'cpu')
print(f"[INFO] Training on: {device}")

## 1. Advanced Configuration

In [ ]:
CONFIG = {
    'data_path': '/kaggle/input/datasets/purvanshjoshi1/healthcare', 
    'img_size': 256,
    'batch_size': 32,
    'epochs': 30,
    'max_lr': 3e-4, 
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'mixup_alpha': 0.2,
    'cutmix_alpha': 1.0,
    'classes': ['glioma', 'meningioma', 'notumor', 'pituitary']
}

## 2. Robust Data Loading (Recursive)

In [ ]:
def load_data(base_path, classes):
    data = []
    print(f"Searching for images in: {base_path}...")
    for cls in classes:
        # Use recursive glob to find classes regardless of nesting
        pattern = os.path.join(base_path, '**', cls, '*')
        paths = glob(pattern, recursive=True)
        # Filter out directories found by glob
        paths = [p for p in paths if os.path.isfile(p)]
        
        if paths:
            print(f"[SUCCESS] Found {len(paths)} images for class '{cls}'")
            for p in paths:
                data.append({'path': p, 'label': cls})
        else:
            print(f"[WARNING] No images found for class '{cls}'")
    
    if not data:
        raise ValueError(f"Critical Error: No images found! Path checked: {base_path}")
        
    df = pd.DataFrame(data)
    label_map = {cls: i for i, cls in enumerate(classes)}
    df['target'] = df['label'].map(label_map)
    return df, label_map

try:
    full_df, label_map = load_data(CONFIG['data_path'], CONFIG['classes'])
    print(f"\nGrand Total: {len(full_df)} images loaded.")
    
    # Stratified Split: Train (80%), Multi-step Val (10%), Test (10%)
    train_df, temp_df = train_test_split(full_df, test_size=0.2, stratify=full_df['target'], random_state=seed)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['target'], random_state=seed)
    
    print(f"Splits: Train({len(train_df)}), Val({len(val_df)}), Test({len(test_df)})")
except Exception as e:
    print(f"\n[ERROR] Loading failed: {e}")

## 3. Advanced Preprocessing & Augmentation
We use **CLAHE** to enhance Contrast and **Albumentations** for a robust pipeline.

In [ ]:
class AdvancedBrainDataset(Dataset):
    def __init__(self, df, transform=None, use_clahe=True):
        self.df = df
        self.transform = transform
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        self.use_clahe = use_clahe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.use_clahe:
            # Apply CLAHE to each channel
            image = np.stack([self.clahe.apply(image[:,:,i]) for i in range(3)], axis=-1)
        
        if self.transform:
            image = self.transform(image=image)['image']
            
        return image, row['target']

train_transform = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    # Fixed parameters for newer Albumentations versions
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(8, 32), hole_width_range=(8, 32), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_ds = AdvancedBrainDataset(train_df, transform=train_transform)
val_ds = AdvancedBrainDataset(val_df, transform=val_transform)
test_ds = AdvancedBrainDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

## 4. Novel Model Architecture
EfficientNet-V2 with a **Hybrid Pooling Head**.

In [ ]:
class HybridEfficientNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        # Backbone: EfficientNet-V2-S
        self.backbone = models.efficientnet_v2_s(weights='IMAGENET1K_V1')
        
        # Remove original classifier
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        # Hybrid Pooling Head
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

model = HybridEfficientNet(num_classes=len(CONFIG['classes'])).to(device)

## 5. Advanced Training with Mixup & CutMix

In [ ]:
# Setup Mixup and CutMix
mixup = v2.MixUp(alpha=CONFIG['mixup_alpha'], num_classes=len(CONFIG['classes']))
cutmix = v2.CutMix(alpha=CONFIG['cutmix_alpha'], num_classes=len(CONFIG['classes']))
cutmix_or_mixup = v2.RandomChoice([mixup, cutmix])

criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['max_lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=CONFIG['max_lr'], 
    steps_per_epoch=len(train_loader), 
    epochs=CONFIG['epochs']
)
scaler = torch.cuda.amp.GradScaler()

best_val_f1 = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(CONFIG['epochs']):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Apply Mixup/CutMix with 50% probability
        if np.random.random() > 0.5:
            images, labels = cutmix_or_mixup(images, labels)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{scheduler.get_last_lr()[0]:.1e}"})
        
    # Validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = (np.array(all_preds) == np.array(all_targets)).mean() * 100
    print(f"[RESULT] Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")
    
    if val_accuracy > best_val_f1:
        best_val_f1 = val_accuracy
        torch.save(model.state_dict(), 'brain_tumor_best.pth')
        print("⭐ Model checkpoint saved!")

## 6. Scientific Evaluation & Confusion Matrix

In [ ]:
model.load_state_dict(torch.load('brain_tumor_best.pth'))
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_targets.extend(labels.numpy())

print("### CLASSIFICATION REPORT ###")
print(classification_report(all_targets, all_preds, target_names=CONFIG['classes']))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CONFIG['classes'], yticklabels=CONFIG['classes'])
plt.title('Confusion Matrix: Clinical Performance Analysis')
plt.ylabel('Ground Truth')
plt.xlabel('AI Prediction')
plt.show()